# Telco Customer Churn - Step 2: Data Cleaning & Feature Preparation

This notebook cleans the raw Telco customer churn dataset and prepares it for downstream modeling.

### Objectives:
1. **Load raw data** (`WA_Fn-UseC_-Telco-Customer-Churn.csv`).
2. **Fix `TotalCharges`**: Convert to numeric and set 11 blank values (`tenure == 0`) to `0.0`.
3. **Drop `customerID`**: Remove non-predictive identifier.
4. **Simplify service categories**: Collapse `"No internet service"` and `"No phone service"` into clean binary `"No"`.
5. **Target encoding**: Convert `Churn` to binary integer (`1` for Yes, `0` for No).
6. **Quality check**: Inspect dtypes, uniqueness, and ensure zero nulls.
7. **Save**: Export cleaned data to `data/cleaned_churn.csv`.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 1. Load the CSV Dataset

In [2]:
# Locate raw CSV relative to notebook or project root
data_path = Path("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
if not data_path.exists():
    data_path = Path("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df = pd.read_csv(data_path)
print(f"Loaded dataset shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

Loaded dataset shape: 7,043 rows, 21 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Fix TotalCharges: Convert to Numeric & Handle Blank Values

In Step 1 exploration, we found 11 rows where `TotalCharges` was whitespace (`' '`). All 11 records have `tenure == 0`, representing newly acquired customers who haven't yet been billed. Rather than dropping them, we set `TotalCharges = 0.0`.

In [3]:
# Coerce invalid string parsing to NaN
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

nan_mask = df["TotalCharges"].isna()
print(f"Number of blank TotalCharges converted to NaN: {nan_mask.sum()}")
print(f"Are all blank rows tenure == 0? {(df.loc[nan_mask, 'tenure'] == 0).all()}")

# Impute 0.0 for new customers with tenure = 0
df["TotalCharges"] = df["TotalCharges"].fillna(0.0)

# Verify dtype and null count
print(f"TotalCharges dtype: {df['TotalCharges'].dtype}")
print(f"Remaining nulls in TotalCharges: {df['TotalCharges'].isna().sum()}")

Number of blank TotalCharges converted to NaN: 11
Are all blank rows tenure == 0? True
TotalCharges dtype: float64
Remaining nulls in TotalCharges: 0


## 3. Drop `customerID`

`customerID` is an arbitrary unique customer identifier without predictive signal.

In [4]:
df = df.drop(columns=["customerID"])
print(f"New shape after dropping customerID: {df.shape}")

New shape after dropping customerID: (7043, 20)


## 4. Simplify Compound Service Categories

Collapse `"No internet service"` and `"No phone service"` into clean `"No"` for:
- `OnlineSecurity`
- `OnlineBackup`
- `DeviceProtection`
- `TechSupport`
- `StreamingTV`
- `StreamingMovies`
- `MultipleLines`

In [5]:
service_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "MultipleLines",
]

replace_map = {
    "No internet service": "No",
    "No phone service": "No",
}

for col in service_cols:
    df[col] = df[col].replace(replace_map)
    print(f"{col:<18} -> unique values: {sorted(df[col].unique().tolist())}")

OnlineSecurity     -> unique values: ['No', 'Yes']
OnlineBackup       -> unique values: ['No', 'Yes']
DeviceProtection   -> unique values: ['No', 'Yes']
TechSupport        -> unique values: ['No', 'Yes']
StreamingTV        -> unique values: ['No', 'Yes']
StreamingMovies    -> unique values: ['No', 'Yes']
MultipleLines      -> unique values: ['No', 'Yes']


## 5. Convert `Churn` to Binary (1 / 0) for Modeling

In [6]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0}).astype("int64")

print("Target 'Churn' distribution:")
print(df["Churn"].value_counts())
print("\nPercentage:")
print(df["Churn"].value_counts(normalize=True) * 100)

Target 'Churn' distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64

Percentage:
Churn
0    73.463013
1    26.536987
Name: proportion, dtype: float64


## 6. Validate Cleaned DataFrame (Dtypes & Null Check)

In [7]:
summary = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Null Count": df.isna().sum(),
    "Unique Values": [df[col].nunique() for col in df.columns],
    "Sample Values": [df[col].unique()[:3].tolist() for col in df.columns]
})
display(summary)

total_nulls = df.isna().sum().sum()
print(f"\nTotal null values across entire dataframe: {total_nulls}")
assert total_nulls == 0, "Data still contains nulls!"

,Data Type,Null Count,Unique Values,Sample Values
gender,str,0,2,"[Female, Male]"
SeniorCitizen,int64,0,2,"[0, 1]"
Partner,str,0,2,"[Yes, No]"
Dependents,str,0,2,"[No, Yes]"
tenure,int64,0,73,"[1, 34, 2]"
PhoneService,str,0,2,"[No, Yes]"
MultipleLines,str,0,2,"[No, Yes]"
InternetService,str,0,3,"[DSL, Fiber optic, No]"
OnlineSecurity,str,0,2,"[No, Yes]"
OnlineBackup,str,0,2,"[Yes, No]"



Total null values across entire dataframe: 0


## 7. Save Cleaned Dataset to `data/cleaned_churn.csv`

In [8]:
output_path = Path("../data/cleaned_churn.csv")
if not output_path.parent.exists():
    output_path = Path("data/cleaned_churn.csv")

df.to_csv(output_path, index=False)
print(f"Successfully saved cleaned data to: {output_path.resolve()}")
print(f"Final shape: {df.shape[0]:,} rows, {df.shape[1]} columns")

Successfully saved cleaned data to: C:\Users\mkc\churn-prediction\data\cleaned_churn.csv
Final shape: 7,043 rows, 20 columns
